In [1]:
from google.colab import drive
drive.mount('/content/drive')

DatasetPath = "/content/drive/MyDrive/dataset/brain-tumor-mri-dataset"
Training_data_path = f"{DatasetPath}/Training"
Testing_data_path = f"{DatasetPath}/Testing"

Mounted at /content/drive


In [2]:
import tensorflow as tf

IMG_SIZE = (256, 256)
BATCH_SIZE = 32  # smaller batch = more gradient updates per epoch, helps a small dataset

Train_data = tf.keras.utils.image_dataset_from_directory(
    Training_data_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

Test_data = tf.keras.utils.image_dataset_from_directory(
    Testing_data_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False  # no need to shuffle validation/test data
)

class_names = Train_data.class_names
print(class_names)

Found 5600 files belonging to 4 classes.
Found 1600 files belonging to 4 classes.
['glioma', 'meningioma', 'notumor', 'pituitary']


In [3]:
AUTOTUNE = tf.data.AUTOTUNE

Train_data = Train_data.cache().shuffle(1000).prefetch(AUTOTUNE)
Test_data = Test_data.cache().prefetch(AUTOTUNE)

In [4]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="data_augmentation")

In [5]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input, Rescaling, Conv2D, MaxPooling2D, BatchNormalization,
    Activation, GlobalAveragePooling2D, Dense, Dropout
)

model = Sequential([
    Input(shape=(256, 256, 3)),

    data_augmentation,          # only active during training
    Rescaling(1./255),          # single normalization, applied to train AND inference

    # Block 1
    Conv2D(32, (3,3), padding="same"), BatchNormalization(), Activation("relu"),
    Conv2D(32, (3,3), padding="same"), BatchNormalization(), Activation("relu"),
    MaxPooling2D(2,2),
    Dropout(0.15),

    # Block 2
    Conv2D(64, (3,3), padding="same"), BatchNormalization(), Activation("relu"),
    Conv2D(64, (3,3), padding="same"), BatchNormalization(), Activation("relu"),
    MaxPooling2D(2,2),
    Dropout(0.20),

    # Block 3
    Conv2D(128, (3,3), padding="same"), BatchNormalization(), Activation("relu"),
    Conv2D(128, (3,3), padding="same"), BatchNormalization(), Activation("relu"),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Block 4
    Conv2D(256, (3,3), padding="same"), BatchNormalization(), Activation("relu"),
    MaxPooling2D(2,2),
    Dropout(0.30),

    GlobalAveragePooling2D(),
    Dense(256, activation="relu"),
    Dropout(0.4),
    Dense(4, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 256, 256, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256, 256, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 256, 256, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 256, 256, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256, 256, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 256, 256, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128, 128, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 128, 128, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 128, 128, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128, 128, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 128, 128, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 64, 64, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 64, 64, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 64, 64, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 651,812 (2.49 MB)

 Trainable params: 650,404 (2.48 MB)

 Non-trainable params: 1,408 (5.50 KB)

In [6]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath="best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=8,
    mode="max",
    restore_best_weights=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
history = model.fit(
    Train_data,
    validation_data=Test_data,
    epochs=40,
    callbacks=[checkpoint, early_stopping, reduce_lr]
)

Epoch 1/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 420ms/step - accuracy: 0.5810 - loss: 1.0088
Epoch 1: val_accuracy improved from None to 0.25000, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
175/175 ━━━━━━━━━━━━━━━━━━━━ 1002s 2s/step - accuracy: 0.6593 - loss: 0.8434 - val_accuracy: 0.2500 - val_loss: 4.4240 - learning_rate: 0.0010
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 420ms/step - accuracy: 0.7399 - loss: 0.6681
Epoch 2: val_accuracy did not improve from 0.25000
175/175 ━━━━━━━━━━━━━━━━━━━━ 78s 443ms/step - accuracy: 0.7482 - loss: 0.6526 - val_accuracy: 0.2419 - val_loss: 3.0733 - learning_rate: 0.0010
Epoch 3/40
100/175 ━━━━━━━━━━━━━━━━━━━━ 31s 419ms/step - accuracy: 0.7863 - loss: 0.5764